<a href="https://colab.research.google.com/github/joaoalexandre14/ex10_avcad/blob/main/ex10AVCAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# 1. Load the dataset
file_path = "EFIplus_medit.csv"
df = pd.read_csv(file_path, sep=';')

# 2. Define the target groups and the NEW strictly environmental variables
target_basins = ['Douro', 'Tejo', 'Mondego', 'Minho']
quant_cols = ['Altitude', 'Actual_river_slope', 'prec_ann_catch', 'temp_ann']

# 3. Filter and clean the data
df_filtered = df[df['Catchment_name'].isin(target_basins)].copy()
# Remove rows with NaNs only in our target columns (drops very few rows!)
df_filtered = df_filtered.dropna(subset=quant_cols)

X = df_filtered[quant_cols]
y = df_filtered['Catchment_name']

# 4. Standardize the variables and run LDA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lda = LinearDiscriminantAnalysis(n_components=2)
lda_components = lda.fit_transform(X_scaled, y)

# Add Discriminant Functions to the dataframe for plotting
df_filtered['LD1'] = lda_components[:, 0]
df_filtered['LD2'] = lda_components[:, 1]

# 5. Create the Interactive Biplot
fig = px.scatter(
    df_filtered, x='LD1', y='LD2', color='Catchment_name',
    hover_data=quant_cols,
    title='Interactive Biplot - LDA (Environmental Variables Only)',
    labels={'Catchment_name': 'Catchment Name'}
)

# Add the variable vectors (Loadings)
loadings = lda.scalings_
scaling_factor = 4 # Adjust this value to scale the vectors visually

for i, feature in enumerate(quant_cols):
    x_end = loadings[i, 0] * scaling_factor
    y_end = loadings[i, 1] * scaling_factor

    # Vector line
    fig.add_shape(
        type='line', x0=0, y0=0, x1=x_end, y1=y_end,
        line=dict(color='black', width=1.5, dash='dot')
    )
    # Variable label
    fig.add_annotation(
        x=x_end, y=y_end, text=feature,
        showarrow=False, xanchor="left", yanchor="bottom",
        font=dict(color="black", size=12)
    )

fig.update_layout(template='plotly_white')

# Opcional: Guarda também como ficheiro HTML para anexo no GitHub/E-learning
fig.write_html("Exercicio_10_Biplot_Interativo.html")

fig.show()